# Experimental results analysis

This notebook presents an analysis of the results obtained from experiments conducted on **sampled data**.

- 📄 **Notebook for data sampling:** [dataset_sampling.ipynb](./dataset_sampling.ipynb)  
- 📂 **Folder with data collection scripts:** [scripts/](../scripts)  

The analysis focuses on evaluating the outcomes of the sampling procedure and interpreting experimental results derived from it.


# Library
This chapter presents the functions and classes that serve as foundational components for the methods and analyses discussed in the subsequent chapters

In [245]:
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import mannwhitneyu
from sklearn.metrics import cohen_kappa_score

Aggregates prediction results by question_id from a list of evaluation runs.

In [246]:
def aggregate_eval_results(results_list: list[dict[str, pd.DataFrame]]) -> list[dict[str, pd.DataFrame]]:
    """
    Args:
        results_list (List[Dict[str, pd.DataFrame]]): 
            Each element is a dictionary of evaluation types (e.g., 'auto-j') 
            mapped to a DataFrame containing columns: ['question_id', 'result', 'ground_truth'].

    Returns:
        List[Dict[str, pd.DataFrame]]: Updated list with grouped and aggregated evaluation results.
    """
    for current_eval_result in results_list:
        for key, value in current_eval_result.items():
            aggregated = value.groupby('question_id').agg({
                'result': lambda x: x.value_counts().idxmax(),
                'ground_truth': 'first'
            })
            aggregated['result_hit'] = aggregated['result'] == aggregated['ground_truth']
            current_eval_result[key] = aggregated
    return results_list

Loads cascaded evaluation results for multiple runs and merges them with a question template.

This function iterates through a sequence of numbered JSON result files (for both auto-j and judgelm evaluations), loads their scores, and attaches these scores to a copy of the provided question template. Each pair of evaluations is stored together in a dictionary and appended to a results list.

In [247]:
import json
from typing import List, Dict, Any


def load_cascaded_eval_results(
        cascad_eval_results_dir: str,
        results_path: str,
        data_type: str,
        vicuna_questions_template: Dict[str, Any],
        num_files: int = 40
) -> List[Dict[str, Dict[str, Any]]]:
    """
    Parameters
    ----------
    cascad_eval_results_dir : str
        Path to the base directory containing cascaded evaluation results.
    results_path : str
        Subdirectory name (relative to `cascad_eval_results_dir`) where the result files are located.
    data_type : str
        The data type identifier (used in file naming).
    vicuna_questions_template : dict
        Template dictionary representing the question data. Must be copyable with `.copy()`.
    num_files : int, optional
        Number of result pairs to load (default is 40).

    Returns
    -------
    List[Dict[str, Dict[str, Any]]]
        A list of dictionaries, where each dictionary contains:
        - 'auto-j': The question template with `score` set to the loaded "auto-j" results.
        - 'judgelm': The question template with `score` set to the loaded "judgelm" results.

    Example
    -------
    >>> results = load_cascaded_eval_results(
    ...     cascad_eval_results_dir=CASCADED_EVAL_RESULTS,
    ...     results_path=RESULTS_PATH,
    ...     data_type=DATA_TYPE,
    ...     vicuna_questions_template=vicuna_vanilla_questions
    ... )
    >>> results[0]['auto-j']
    {'question': '...', 'score': [...]}
    """
    cascaded_eval_result_list = []

    for i in range(1, num_files + 1):
        auto_j_file = f"{cascad_eval_results_dir}/{results_path}/{data_type}/{i}-auto-j-{data_type}-final.json"
        judgelm_file = f"{cascad_eval_results_dir}/{results_path}/{data_type}/{i}-judgelm-{data_type}-final.json"

        with open(auto_j_file, "r") as f:
            auto_j_results = json.loads(f.read().strip())

        with open(judgelm_file, "r") as f:
            judgelm_results = json.loads(f.read().strip())

        auto_j = vicuna_questions_template.copy()
        auto_j['score'] = auto_j_results

        judgelm = vicuna_questions_template.copy()
        judgelm['score'] = judgelm_results

        cascaded_eval_result_list.append({
            'auto-j': auto_j,
            'judgelm': judgelm
        })

    return cascaded_eval_result_list


Following class functionality:

- This class performs statistical hypothesis testing between two numerical samples.
- It automatically chooses between a T-test and Mann-Whitney U test based on normality and variance checks.

In [248]:
class TwoSampleStatisticalTests:
    def __init__(self, first_dataset: pd.Series, second_dataset: pd.Series, significance_level=0.05):
        """
        Initialize the class with two datasets and a significance level.

        :param first_dataset: First sample (pandas Series)
        :param second_dataset: Second sample (pandas Series)
        :param significance_level: Alpha threshold for hypothesis testing (e.g., 0.05)
        """
        self._first_dataset = first_dataset
        self._second_dataset = second_dataset
        self._significance_level = significance_level

    def _check_normality(self) -> bool:
        """
        Test for normal distribution using Shapiro-Wilk’s test.

        H0: The data is normally distributed  
        Ha: The data is not normally distributed

        :return: True if both datasets are normally distributed, else False
        """
        _, pvalue_first = stats.shapiro(self._first_dataset) if len(self._first_dataset) <= 5000 else stats.normaltest(
            self._first_dataset)
        _, pvalue_second = stats.shapiro(self._second_dataset) if len(
            self._second_dataset) <= 5000 else stats.normaltest(self._second_dataset)

        print(f"The result of the p-value when checking the normality for the first dataset: {pvalue_first}")
        print(f"The result of the p-value when checking the normality for the second dataset: {pvalue_second}")

        return pvalue_first >= self._significance_level and pvalue_second >= self._significance_level

    def _check_variance_homogeneity(self) -> bool:
        """
        Test for homogeneity of variances using Levene's test.

        H0: Variances are equal (homogeneous)  
        Ha: Variances are different

        :return: True if variances are equal, else False
        """
        _, pvalue = stats.levene(self._first_dataset, self._second_dataset)

        print(f"The result of the p-value when checking the variance uniform: {pvalue}")

        return pvalue >= self._significance_level

    def _t_test(self, is_one_tailed: bool = False) -> float:
        """
        Perform an independent two-sample T-test assuming equal variances.

        :param is_one_tailed: If True, calculates one-tailed p-value.
        :return: p-value from T-test
        """
        t_stat, p_value = stats.ttest_ind(self._first_dataset, self._second_dataset, equal_var=True)

        # Adjust for one-tailed test if requested
        return p_value if not is_one_tailed else (p_value / 2 if t_stat > 0 else 1 - (p_value / 2))

    def _mann_whitney_test(self, is_one_tailed: bool = False) -> float:
        """
        Perform the Mann-Whitney U test (non-parametric alternative to T-test).

        :param is_one_tailed: If True, uses 'greater' alternative.
        :return: p-value from Mann-Whitney test
        """
        alternative = 'greater' if is_one_tailed else 'two-sided'
        _, p_value = mannwhitneyu(self._first_dataset, self._second_dataset, alternative=alternative)
        return p_value

    def test_two_numerical_samples(self, is_one_tailed: bool = False):
        """
        Runs the appropriate hypothesis test between two numerical samples.

        - If both samples are normally distributed AND variances are equal: T-test is used  
        - Otherwise: Mann-Whitney U test is used

        Prints the test used, the final p-value, and the hypothesis decision.
        """
        # Step 1: Check distribution
        is_normally_distributed = self._check_normality()

        # Step 2: Check variance homogeneity
        is_variance_homogeneous = self._check_variance_homogeneity()

        # Step 3: Choose test based on assumptions
        if is_normally_distributed and is_variance_homogeneous:
            print("T-test was chosen")
            p_value = self._t_test(is_one_tailed)
        else:
            print("Mann-Whitney was chosen")
            p_value = self._mann_whitney_test(is_one_tailed)

        # Step 4: Report results
        print(f"Final p-value: {p_value}")

        if p_value <= self._significance_level:
            print("H0 has been rejected, Ha has been accepted")
        else:
            print("H0 was not rejected")

Reverse score

In [249]:
def reverse_row(row):
    first_score, second_score = row['score']
    is_reversed = row['is_reversed']

    if is_reversed:
        row['score'] = [second_score, first_score]
        row['text_gpt'], row['text_vicuna'] = row['text_vicuna'], row['text_gpt']

    return row

Get top `ratio` indexes

In [250]:
def get_top_half_indices(relia_scores, ratio=0.5):
    sorted_indices = np.argsort(-np.array(relia_scores))
    top_half_indices = sorted_indices[:int(len(sorted_indices) * ratio)]

    return list(top_half_indices)

Gather the final metrics (Accuracy and kappa) + save each observation for the further statistical analysis

In [251]:
def gather_final_metrics(result_list, hit_feature="result_hit", result_feature="result",
                         ground_truth_feature="ground_truth"):
    acc_list, kappa_list = np.array([]), np.array([])
    
    for res in result_list:
        acc_list = np.append(acc_list, res[hit_feature].mean() * 100)
        kappa_list = np.append(kappa_list, cohen_kappa_score(res[ground_truth_feature], res[result_feature]))

    return acc_list.mean(), kappa_list.mean(), acc_list, kappa_list

Extract the ground truth and result

In [252]:
def extract_result(row, ground_truth, win_label="CHATGPT", lose_label="VICUNA13B", tie_label="TIE"):
    first_score, second_score = row['score']

    row['result'] = win_label if round(first_score, 2) > round(second_score, 2) else lose_label if round(first_score,
                                                                                                         2) < round(
        second_score, 2) else tie_label
    row['ground_truth'] = ground_truth[row.name - 1]
    row['result_hit'] = row['result'] == row['ground_truth']

    return row

ID's extraction of less confident results

In [253]:
def extract_non_confident_indices(result_df, ratio=0.5):
    relia_scores = result_df['entropy'].to_numpy()

    sorted_indices = np.argsort(-relia_scores)
    top_half_indices = sorted_indices[:int(len(sorted_indices) * ratio)]

    top_df_indices = result_df.index[top_half_indices]

    return set(result_df.index) - set(top_df_indices)

# Ground-truth reading
This chapter is dedicated for reading the ground-truth for experiments related to the Calibrated Judge and Calibrated scorer (ground-truth from Vicuna and adjusted version for CS)

In [254]:
with open('../datasets/vicuna/ground-truth/cj_ground_truth.txt', 'r', encoding='utf-8') as f:
    text_lines = f.read().splitlines()

len(text_lines)

80

Ground-truth for Calibrated Judge + Calibrated Scorer tests - accept TIE as a baseline

In [255]:
with open('../datasets/vicuna/ground-truth/cs_cj_ground_truth.txt', 'r', encoding='utf-8') as f:
    lc_text_lines = f.read().splitlines()

len(lc_text_lines)

80

# Baseline and MEC + BPC results
Read the baseline and MEC+BPC results, extract accuracy metrics and run the statistical tests

The folder with the results

In [256]:
BASELINE_MEC_RESULTS_PATH = "../gathered_data/baseline_mec_results"

Read the results from all **40** tries

In [257]:
sampled_result_list = []

for it in range(1, 41):
    current_judgement_result = {
        'gpt35_k_1_bpc_0_t_1': pd.read_json(
            f'{BASELINE_MEC_RESULTS_PATH}/{it}/review_gpt35_vicuna_gpt-3.5-turbo_mec1_bpc0.jsonl',
            lines=True).set_index(
            'question_id'),
        'gpt4_k_1_bpc_0_t_1': pd.read_json(
            f'{BASELINE_MEC_RESULTS_PATH}/{it}/review_gpt35_vicuna_gpt-4_mec1_bpc0.jsonl',
            lines=True).set_index(
            'question_id'),
        'gpt35_k_3_bpc_1_t_1': pd.read_json(
            f'{BASELINE_MEC_RESULTS_PATH}/{it}/review_gpt35_vicuna_gpt-3.5-turbo_mec3_bpc1.jsonl',
            lines=True).set_index(
            'question_id'),
        'gpt4_k_3_bpc_1_t_1': pd.read_json(
            f'{BASELINE_MEC_RESULTS_PATH}/{it}/review_gpt35_vicuna_gpt-4_mec3_bpc1.jsonl',
            lines=True).set_index(
            'question_id'),
    }

    sampled_result_list.append(current_judgement_result)

sampled_result_list[0]['gpt4_k_3_bpc_1_t_1']

,question,review,review_bpc,cost,score
question_id,,,,,
5,Can you explain the basics of quantum computing?,[Evaluation evidence: Both assistants provided...,[Evaluation evidence: Both assistants effectiv...,0.10722,"[8.75, 9.666666666666666]"
14,How do language and cultural barriers affect t...,[Evaluation evidence: Both assistants provided...,[Evaluation evidence: Both assistants provided...,0.09276,"[8.5, 9.666666666666666]"
29,"As a space colonist on Mars, describe your dai...",[Evaluation evidence: Both Assistants have pro...,[Evaluation evidence: Both assistants provided...,0.12096,"[9.333333333333334, 8.916666666666666]"
37,Why do some people enjoy the sensation of bein...,[Evaluation evidence: Both assistants provided...,[Evaluation evidence: Both responses are preci...,0.08526,"[8.666666666666666, 9.666666666666666]"
47,How many snowflakes fall during a typical wint...,[Evaluation evidence: Both assistants provided...,[Evaluation evidence: Both Assistant 1 and Ass...,0.11928,"[8.583333333333334, 8.416666666666666]"
56,What if Alan Turing had not cracked the Enigma...,[Evaluation evidence: Both the assistants did ...,[Evaluation evidence: Both assistants provided...,0.09318,"[9.916666666666666, 9.416666666666666]"
63,Implement a regular expression in Python to va...,[Evaluation evidence: Both assistants provided...,[Evaluation evidence: Both Assistant 1 and Ass...,0.09354,"[9.5, 8.833333333333334]"
69,Solve for x in the equation 3x + 10 = 5(x - 2).,[Evaluation evidence: Assistant 1 provided a s...,[Evaluation evidence: Assistant 1's response i...,0.08904,"[10.0, 1.5]"
77,Compose an engaging travel blog post about a r...,[Evaluation evidence: Both Assistant 1 and Ass...,[Evaluation evidence: Both assistants provided...,0.13308,"[9.5, 9.916666666666666]"


Extract the final judgement result and its match with the **ground-truth**

In [258]:
for key in sampled_result_list[0].keys():
    for res in sampled_result_list:
        res[key] = res[key].apply(lambda row: extract_result(row, text_lines), axis=1)

sampled_result_list[2]['gpt4_k_1_bpc_0_t_1']

,question,review,review_bpc,cost,score,result,ground_truth,result_hit
question_id,,,,,,,,
7,How can I develop my critical thinking skills?,[Evaluation evidence: Both assistants provided...,[],0.03177,"[9.0, 10.0]",VICUNA13B,CHATGPT,False
17,How do vaccinations work to protect individual...,[Evaluation evidence: Both assistants provided...,[],0.02817,"[9.0, 10.0]",VICUNA13B,VICUNA13B,True
27,Pretend to be a world-famous chef. How would y...,[Evaluation evidence: Both assistants provided...,[],0.03441,"[9.0, 8.0]",CHATGPT,CHATGPT,True
38,How can observing the behavior of other people...,[Evaluation evidence: Both assistants provided...,[],0.02889,"[10.0, 8.0]",CHATGPT,CHATGPT,True
45,How many text messages are sent globally in a ...,[Evaluation evidence: Assistant 1 provided a d...,[],0.03018,"[9.0, 6.0]",CHATGPT,CHATGPT,True
57,What if the Suez Canal had never been construc...,[Evaluation evidence: Both assistants provided...,[],0.02751,"[8.0, 9.0]",VICUNA13B,VICUNA13B,True
66,Implement a queue data structure using two sta...,[Evaluation evidence: Assistant 1 provided a c...,[],0.03570,"[10.0, 2.0]",CHATGPT,CHATGPT,True
69,Solve for x in the equation 3x + 10 = 5(x - 2).,[Evaluation evidence: Assistant 1 provided a c...,[],0.02874,"[10.0, 2.0]",CHATGPT,CHATGPT,True
76,Write a script for a YouTube video exploring t...,[Evaluation evidence: Both assistants provided...,[],0.03771,"[9.0, 8.0]",CHATGPT,CHATGPT,True


Extract the **accuracy** and **kappa** metrics from all the 40 attempts

In [259]:
sampled_accuracy_map = {}
sampled_kappa_map = {}

sampled_acc_list = {}
sampled_kappa_list = {}

for key in sampled_result_list[0].keys():
    current_result = list(map(lambda x: x[key], sampled_result_list))
    

    sampled_accuracy_map[key], sampled_kappa_map[key], sampled_acc_list[key], sampled_kappa_list[
        key] = gather_final_metrics(current_result)

sampled_accuracy_map

{'gpt35_k_1_bpc_0_t_1': 44.166666666666664,
 'gpt4_k_1_bpc_0_t_1': 88.33333333333334,
 'gpt35_k_3_bpc_1_t_1': 79.44444444444444,
 'gpt4_k_3_bpc_1_t_1': 46.94444444444444}

Two statistical tests were conducted to determine whether **MEC + BPC** achieved significantly higher results than the **baseline** in terms of **accuracy** and **kappa**, using **GPT-4** as the baseline (got the **p-values** here). 

- In both cases, the null hypothesis H_0 was not rejected
- **MEC + BPC** did not demonstrate a statistically significant improvement over **GPT-4** in either metric.

In [260]:
gpt_4_tester_acc = TwoSampleStatisticalTests(sampled_acc_list['gpt4_k_3_bpc_1_t_1'],
                                             sampled_acc_list['gpt4_k_1_bpc_0_t_1'], )
gpt_4_tester_kappa = TwoSampleStatisticalTests(sampled_kappa_list['gpt4_k_3_bpc_1_t_1'],
                                               sampled_kappa_list['gpt4_k_1_bpc_0_t_1'])

print("Accuracy\n")
gpt_4_tester_acc.test_two_numerical_samples(is_one_tailed=True)

print("\nKappa score\n")
gpt_4_tester_kappa.test_two_numerical_samples(is_one_tailed=True)

Accuracy

The result of the p-value when checking the normality for the first dataset: 8.268180079220535e-05
The result of the p-value when checking the normality for the second dataset: 3.2512410563000877e-13
The result of the p-value when checking the variance uniform: 1.0646977938864353e-05
Mann-Whitney was chosen
Final p-value: 0.9999999999999999
H0 was not rejected

Kappa score

The result of the p-value when checking the normality for the first dataset: 0.0011599938206142493
The result of the p-value when checking the normality for the second dataset: 3.569372791683109e-13
The result of the p-value when checking the variance uniform: 6.875942870679636e-07
Mann-Whitney was chosen
Final p-value: 0.9999999999999999
H0 was not rejected


Two statistical tests were conducted to determine whether **MEC + BPC** achieved significantly higher results than the **baseline** in terms of **accuracy** and **kappa**, using **GPT-3.5-turbo** as the baseline. 

- In both cases, the null hypothesis H_0 was rejected
- **MEC + BPC** demonstrated a statistically significant improvement over **GPT-3.5-turbo** in either metric.

In [261]:
gpt_35_tester_acc = TwoSampleStatisticalTests(sampled_acc_list['gpt35_k_3_bpc_1_t_1'],
                                              sampled_acc_list['gpt35_k_1_bpc_0_t_1'])
gpt_35_tester_kappa = TwoSampleStatisticalTests(sampled_kappa_list['gpt35_k_3_bpc_1_t_1'],
                                                sampled_kappa_list['gpt35_k_1_bpc_0_t_1'])

print("ACC\n")
gpt_35_tester_acc.test_two_numerical_samples(is_one_tailed=True)

print("\nKAPPA\n")
gpt_35_tester_kappa.test_two_numerical_samples(is_one_tailed=True)

ACC

The result of the p-value when checking the normality for the first dataset: 5.977871270659952e-07
The result of the p-value when checking the normality for the second dataset: 1.1298991811008934e-05
The result of the p-value when checking the variance uniform: 0.11756183162228954
Mann-Whitney was chosen
Final p-value: 1.28089233092466e-15
H0 has been rejected, Ha has been accepted

KAPPA

The result of the p-value when checking the normality for the first dataset: 1.445952369266414e-05
The result of the p-value when checking the normality for the second dataset: 0.07888856378526432
The result of the p-value when checking the variance uniform: 0.008771681577263369
Mann-Whitney was chosen
Final p-value: 4.867435269562136e-15
H0 has been rejected, Ha has been accepted


Results description for the final table

In [262]:
SAMPLED_DESCRIPTION_MAP = {
    'gpt35_k_1_bpc_0_t_1': {
        'judge': 'GPT-3.5',
        'description': 'EC (k = 1)'
    },
    'gpt4_k_1_bpc_0_t_1': {
        'judge': 'GPT-4',
        'description': 'EC (k = 1)'
    },
    'gpt35_k_3_bpc_1_t_1': {
        'judge': 'GPT-3.5 + MEC + BPC',
        'description': 'MEC (k = 3) + BPC (k = 3)'
    },
    'gpt4_k_3_bpc_1_t_1': {
        'judge': 'GPT-4 + MEC + BPC',
        'description': 'MEC (k = 3) + BPC (k = 3)'
    },
}

Final table with **accuracy** and **kappa** for **baselines** and **MEC+BPC** with GPT-4 and GPT-3.5-turbo as baselines

In [263]:
final_table = pd.DataFrame({
    "judge": [v["judge"] for v in SAMPLED_DESCRIPTION_MAP.values()],
    "description": [v["description"] for v in SAMPLED_DESCRIPTION_MAP.values()],
    "accuracy": sampled_accuracy_map.values(),
    "kappa": sampled_kappa_map.values(),
}, index=sampled_accuracy_map.keys())

final_table

,judge,description,accuracy,kappa
gpt35_k_1_bpc_0_t_1,GPT-3.5,EC (k = 1),44.166667,0.066273
gpt4_k_1_bpc_0_t_1,GPT-4,EC (k = 1),88.333333,0.716802
gpt35_k_3_bpc_1_t_1,GPT-3.5 + MEC + BPC,MEC (k = 3) + BPC (k = 3),79.444444,0.639853
gpt4_k_3_bpc_1_t_1,GPT-4 + MEC + BPC,MEC (k = 3) + BPC (k = 3),46.944444,0.124227


![Paper results](../figures/fair_eval_experiment.png)

# Accuracy distribution around the categories for baseline / MEC + BPC

The following chapter is designed to evaluate how the baseline and MEC + BPC approaches perform across different question types in the dataset, using GPT-4 and GPT-3.5-turbo.

In [264]:
SAMPLED_QUESTIONS_PATH = "../datasets/vicuna/sampled_data/cj_sampled/questions"

In [265]:
sampled_question = {
    'gpt35_k_1_bpc_0_t_1': pd.read_json(f"{SAMPLED_QUESTIONS_PATH}/0.jsonl", lines=True),
    'gpt4_k_1_bpc_0_t_1': pd.read_json(f"{SAMPLED_QUESTIONS_PATH}/1.jsonl", lines=True),
    'gpt35_k_3_bpc_1_t_1': pd.read_json(f"{SAMPLED_QUESTIONS_PATH}/2.jsonl", lines=True),
    'gpt4_k_3_bpc_1_t_1': pd.read_json(f"{SAMPLED_QUESTIONS_PATH}/3.jsonl", lines=True)
}

sampled_question['gpt35_k_1_bpc_0_t_1']

,question_id,text,category
0,5,Can you explain the basics of quantum computing?,generic
1,11,What are some potential implications of using ...,knowledge
2,28,You are a mountain climber reaching the summit...,roleplay
3,34,How can you determine if a person is genuinely...,common-sense
4,43,How many lightning strikes occur on Earth each...,fermi
5,53,What if the Black Death had not occurred in th...,counterfactual
6,64,Write a program to find the nth Fibonacci numb...,coding
7,70,"If the endpoints of a line segment are (2, -2)...",math
8,80,"Write a symphony concert review, discussing th...",writing


Creating a template for the final table where x-axis == **mode** (baseline or MEC+BPC + version of GPT)

In [266]:
topic_comparison_base = {}

for key in sampled_question.keys():
    key_name = SAMPLED_DESCRIPTION_MAP[key]['judge']

    if key_name not in topic_comparison_base:
        topic_comparison_base[key_name] = {}
    for category in sampled_question['gpt35_k_1_bpc_0_t_1']['category'].unique():
        topic_comparison_base[key_name][category] = 0

topic_comparison_table = pd.DataFrame(topic_comparison_base)
topic_comparison_table

,GPT-3.5,GPT-4,GPT-3.5 + MEC + BPC,GPT-4 + MEC + BPC
generic,0,0,0,0
knowledge,0,0,0,0
roleplay,0,0,0,0
common-sense,0,0,0,0
fermi,0,0,0,0
counterfactual,0,0,0,0
coding,0,0,0,0
math,0,0,0,0
writing,0,0,0,0


Counting the amount of times when each mode has succeeded on each topic (sampling is 1 question per topic)

In [267]:
def process_row(row, comp_table, feature_name):
    comp_table.at[row['category'], feature_name] += 1 if row['result_hit'] else 0


for sample in sampled_result_list:
    for key in sample.keys():
        current_df = sample[key]
        current_questions = sampled_question[key]

        feature_name = SAMPLED_DESCRIPTION_MAP[key]['judge']

        merged_df = pd.merge(current_df, current_questions, left_index=True, right_on="question_id")
        merged_df.apply(lambda row: process_row(row, topic_comparison_table, feature_name), axis=1)

topic_comparison_table

,GPT-3.5,GPT-4,GPT-3.5 + MEC + BPC,GPT-4 + MEC + BPC
generic,21,0,39,40
knowledge,2,40,40,0
roleplay,0,39,12,0
common-sense,0,40,40,40
fermi,40,40,40,19
counterfactual,2,39,2,1
coding,37,40,39,24
math,40,40,40,40
writing,17,40,34,5


Make the results relative and in %:
- **GPT-3.5**: MEC + BPC improves accuracy across all categories, including weaker areas like knowledge, roleplay, and writing
- **GPT-4**: Gains in open-ended tasks (generic, common-sense) but notable drops in specialized tasks (roleplay, counterfactual, fermi, writing)
- **Cause**: Likely misalignment with GPT-4’s internal bias controls, adding noise instead of reducing bias

In [268]:
topic_comparison_table = topic_comparison_table / 40 * 100
topic_comparison_table = topic_comparison_table.applymap(lambda x: f"{x}%" if pd.notnull(x) else "")
topic_comparison_table

/var/folders/1r/q7sz92cx6mvgt9xkk4_68cn00000gn/T/ipykernel_6873/3098987464.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  topic_comparison_table = topic_comparison_table.applymap(lambda x: f"{x}%" if pd.notnull(x) else "")


,GPT-3.5,GPT-4,GPT-3.5 + MEC + BPC,GPT-4 + MEC + BPC
generic,52.5%,0.0%,97.5%,100.0%
knowledge,5.0%,100.0%,100.0%,0.0%
roleplay,0.0%,97.5%,30.0%,0.0%
common-sense,0.0%,100.0%,100.0%,100.0%
fermi,100.0%,100.0%,100.0%,47.5%
counterfactual,5.0%,97.5%,5.0%,2.5%
coding,92.5%,100.0%,97.5%,60.0%
math,100.0%,100.0%,100.0%,100.0%
writing,42.5%,100.0%,85.0%,12.5%


# CascadeEval extraction with GPT-3.5 as a baseline
This chapter extracts evaluation data obtained using the **CascadedEval** approach, computes the control metrics (**accuracy** and **Cohen’s kappa**), and compiles a summary table for the **GPT-3.5-turbo (baseline)** configuration.

Define folder of data type from merged sampling dataset

In [269]:
DATA_TYPE = 'vicuna'

Obtain the aggregated sampled subset from the Vicuna dataset

In [270]:
vicuna_vanilla_questions = pd.read_json(f'../datasets/vicuna/sampled_data/cj_sampled_merged/vanilla-{DATA_TYPE}.json')
vicuna_vanilla_questions['ground_truth'] = vicuna_vanilla_questions['question_id'].apply(lambda x: text_lines[x - 1])
vicuna_vanilla_questions = vicuna_vanilla_questions.set_index('question_id')

vicuna_vanilla_questions

,text_vicuna,text_gpt,question_text,category,ground_truth
question_id,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,VICUNA13B
11,Using a single-use plastic bottle versus a reu...,Using a single-use plastic bottle has negative...,What are some potential implications of using ...,knowledge,TIE
28,"As a language model AI, I don't have personal ...","I'm sorry, but as an AI language model, I am n...",You are a mountain climber reaching the summit...,roleplay,CHATGPT
34,It can be difficult to determine if a person i...,There are several ways to determine if a perso...,How can you determine if a person is genuinely...,common-sense,TIE
43,It is difficult to give an exact number of how...,"According to research, it is estimated that th...",How many lightning strikes occur on Earth each...,fermi,CHATGPT
53,"The Black Death, also known as the Plague, was...",If the Black Death had not occurred in the 14t...,What if the Black Death had not occurred in th...,counterfactual,TIE
64,Here is an example of a program in Python that...,Certainly! Here's an example Python program us...,Write a program to find the nth Fibonacci numb...,coding,CHATGPT
70,The length of the line segment is the distance...,To find the length of a line segment with two ...,"If the endpoints of a line segment are (2, -2)...",math,CHATGPT
80,"The concert was a true spectacle of sound, fea...",I'm happy to assist you in writing a symphony ...,"Write a symphony concert review, discussing th...",writing,CHATGPT


Define the folder containing **CascadedEval** results for the **sampled portion** of the dataset for case when **GPT-3.5-turbo** is the baseline configuration.

In [271]:
CASCADED_EVAL_RESULTS = '../gathered_data/cascaded-eval-results'

RESULTS_PATH = 'final_results'

Extract final judgement results (either Auto-J / JudgeLM or baseline that is pure GPT-3.5-turbo) for each question to calculate **accuracy** and **kappa**

In [272]:
cascaded_eval_result_list = load_cascaded_eval_results(
    cascad_eval_results_dir=CASCADED_EVAL_RESULTS,
    results_path=RESULTS_PATH,
    data_type=DATA_TYPE,
    vicuna_questions_template=vicuna_vanilla_questions
)

cascaded_eval_result_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,ground_truth,score
question_id,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,VICUNA13B,"[0, 1]"
11,Using a single-use plastic bottle versus a reu...,Using a single-use plastic bottle has negative...,What are some potential implications of using ...,knowledge,TIE,"[9.0, 8.0]"
28,"As a language model AI, I don't have personal ...","I'm sorry, but as an AI language model, I am n...",You are a mountain climber reaching the summit...,roleplay,CHATGPT,"[0, 1]"
34,It can be difficult to determine if a person i...,There are several ways to determine if a perso...,How can you determine if a person is genuinely...,common-sense,TIE,"[0, 1]"
43,It is difficult to give an exact number of how...,"According to research, it is estimated that th...",How many lightning strikes occur on Earth each...,fermi,CHATGPT,"[1, 0]"
53,"The Black Death, also known as the Plague, was...",If the Black Death had not occurred in the 14t...,What if the Black Death had not occurred in th...,counterfactual,TIE,"[8.0, 7.0]"
64,Here is an example of a program in Python that...,Certainly! Here's an example Python program us...,Write a program to find the nth Fibonacci numb...,coding,CHATGPT,"[9.0, 5.0]"
70,The length of the line segment is the distance...,To find the length of a line segment with two ...,"If the endpoints of a line segment are (2, -2)...",math,CHATGPT,"[9.0, 6.0]"
80,"The concert was a true spectacle of sound, fea...",I'm happy to assist you in writing a symphony ...,"Write a symphony concert review, discussing th...",writing,CHATGPT,"[9.0, 8.0]"


Extract final judgement results and hit with the ground-truth

In [273]:
for key in cascaded_eval_result_list[0].keys():
    for res in cascaded_eval_result_list:
        res[key] = res[key].apply(lambda row: extract_result(row, text_lines), axis=1)

cascaded_eval_result_list[5]['auto-j']

,text_vicuna,text_gpt,question_text,category,ground_truth,score,result,result_hit
question_id,,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,VICUNA13B,"[0, 1]",VICUNA13B,True
11,Using a single-use plastic bottle versus a reu...,Using a single-use plastic bottle has negative...,What are some potential implications of using ...,knowledge,TIE,"[8.0, 6.0]",CHATGPT,False
28,"As a language model AI, I don't have personal ...","I'm sorry, but as an AI language model, I am n...",You are a mountain climber reaching the summit...,roleplay,CHATGPT,"[0, 1]",VICUNA13B,False
34,It can be difficult to determine if a person i...,There are several ways to determine if a perso...,How can you determine if a person is genuinely...,common-sense,TIE,"[0, 1]",VICUNA13B,False
43,It is difficult to give an exact number of how...,"According to research, it is estimated that th...",How many lightning strikes occur on Earth each...,fermi,CHATGPT,"[1, 0]",CHATGPT,True
53,"The Black Death, also known as the Plague, was...",If the Black Death had not occurred in the 14t...,What if the Black Death had not occurred in th...,counterfactual,TIE,"[8.0, 6.0]",CHATGPT,False
64,Here is an example of a program in Python that...,Certainly! Here's an example Python program us...,Write a program to find the nth Fibonacci numb...,coding,CHATGPT,"[9.0, 6.0]",CHATGPT,True
70,The length of the line segment is the distance...,To find the length of a line segment with two ...,"If the endpoints of a line segment are (2, -2)...",math,CHATGPT,"[9.0, 5.0]",CHATGPT,True
80,"The concert was a true spectacle of sound, fea...",I'm happy to assist you in writing a symphony ...,"Write a symphony concert review, discussing th...",writing,CHATGPT,"[9.0, 8.0]",CHATGPT,True


Extract mode of the result in the experiment results

In [274]:
cascaded_eval_result_list = aggregate_eval_results(cascaded_eval_result_list)
cascaded_eval_result_list[5]['auto-j']

,result,ground_truth,result_hit
question_id,,,
5,VICUNA13B,VICUNA13B,True
11,CHATGPT,TIE,False
28,VICUNA13B,CHATGPT,False
34,VICUNA13B,TIE,False
43,CHATGPT,CHATGPT,True
53,CHATGPT,TIE,False
64,CHATGPT,CHATGPT,True
70,CHATGPT,CHATGPT,True
80,CHATGPT,CHATGPT,True


Get accuracy and kappa for each of **40** repeated experiments

In [275]:
cascaded_accuracy_map = {}
cascaded_kappa_map = {}

for key in cascaded_eval_result_list[0].keys():
    current_result = list(map(lambda x: x[key], cascaded_eval_result_list))
    cascaded_accuracy_map[key], cascaded_kappa_map[key], _, _ = gather_final_metrics(current_result)

cascaded_accuracy_map

{'auto-j': 50.96618357487922, 'judgelm': 56.763285024154605}

Decode information about the experiments

In [276]:
CASCADED_DESCRIPTION_MAP = {
    'auto-j': {
        'judge': 'Auto-J (CascadedEval with GPT-3.5)',
        'description': 'CascadedEval + EC (k = 1) + GPT-3.5'
    },
    'judgelm': {
        'judge': 'JudgeLM (CascadedEval with GPT-3.5)',
        'description': 'CascadedEval + EC (k = 1) + GPT-3.5'
    },
}

Gather everything in the final table (together with the previous steps)

In [277]:
cascaded_final_table = pd.DataFrame({
    "judge": [v["judge"] for v in CASCADED_DESCRIPTION_MAP.values()],
    "description": [v["description"] for v in CASCADED_DESCRIPTION_MAP.values()],
    "accuracy": cascaded_accuracy_map.values(),
    "kappa": cascaded_kappa_map.values(),
}, index=cascaded_accuracy_map.keys())

cascaded_final_table = pd.concat([cascaded_final_table, final_table], ignore_index=True)
cascaded_final_table

,judge,description,accuracy,kappa
0,Auto-J (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,50.966184,0.211477
1,JudgeLM (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,56.763285,0.216264
2,GPT-3.5,EC (k = 1),44.166667,0.066273
3,GPT-4,EC (k = 1),88.333333,0.716802
4,GPT-3.5 + MEC + BPC,MEC (k = 3) + BPC (k = 3),79.444444,0.639853
5,GPT-4 + MEC + BPC,MEC (k = 3) + BPC (k = 3),46.944444,0.124227


# CascadedEval extraction with GPT-3.5 with MEC
This chapter extracts evaluation data obtained using the **CascadedEval** approach, computes the control metrics (**accuracy** and **Cohen’s kappa**), and compiles a summary table for the **GPT-3.5-turbo (MEC + BPC)** configuration.

Define folder of data type from merged sampling dataset


In [278]:
DATA_TYPE = 'vicuna-mec'

Obtain the aggregated sampled subset from the Vicuna dataset

In [279]:
vicuna_mec_questions = pd.read_json(f'../datasets/vicuna/sampled_data/cj_sampled_merged/mec-bpc-vicuna.json')
vicuna_mec_questions['ground_truth'] = vicuna_mec_questions['question_id'].apply(lambda x: text_lines[x - 1])
vicuna_mec_questions = vicuna_mec_questions.set_index('question_id')

vicuna_mec_questions

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth
question_id,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B


Extract final judgement results (either Auto-J / JudgeLM or MEC+BPC with GPT-3.5-turbo) for each question to calculate **accuracy** and **kappa**

In [280]:
mec_cascaded_eval_result_list = load_cascaded_eval_results(
    cascad_eval_results_dir=CASCADED_EVAL_RESULTS,
    results_path=RESULTS_PATH,
    data_type=DATA_TYPE,
    vicuna_questions_template=vicuna_mec_questions
)

mec_cascaded_eval_result_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth,score
question_id,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[1, 0]"
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[1, 0]"
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[1, 0]"
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]"
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]"
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]"


Reverse rows that are under BPC

In [281]:
for key in mec_cascaded_eval_result_list[0].keys():
    for res in mec_cascaded_eval_result_list:
        res[key] = res[key].apply(reverse_row, axis=1)

mec_cascaded_eval_result_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth,score
question_id,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]"
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]"
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]"
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]"


Extract final judgement results and hit with the ground-truth

In [282]:
for key in mec_cascaded_eval_result_list[0].keys():
    for res in mec_cascaded_eval_result_list:
        res[key] = res[key].apply(lambda row: extract_result(row, text_lines), axis=1)

mec_cascaded_eval_result_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth,score,result,result_hit
question_id,,,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]",VICUNA13B,True
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]",VICUNA13B,True
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]",VICUNA13B,True
18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,knowledge,False,VICUNA13B,"[0, 1]",VICUNA13B,True


Extract mode of the result in the experiment results

In [283]:
mec_cascaded_eval_result_list = aggregate_eval_results(mec_cascaded_eval_result_list)
mec_cascaded_eval_result_list[0]['auto-j']

,result,ground_truth,result_hit
question_id,,,
5,VICUNA13B,VICUNA13B,True
18,VICUNA13B,VICUNA13B,True
26,CHATGPT,CHATGPT,True
40,VICUNA13B,VICUNA13B,True
45,CHATGPT,CHATGPT,True
53,VICUNA13B,TIE,False
63,VICUNA13B,VICUNA13B,True
70,CHATGPT,CHATGPT,True
76,CHATGPT,CHATGPT,True


Get accuracy and kappa for each of **40** repeated experiments

In [284]:
mec_cascaded_accuracy_map = {}
mec_cascaded_kappa_map = {}

mec_cascaded_accuracy_list = {}
mec_cascaded_kappa_list = {}

for key in mec_cascaded_eval_result_list[0].keys():
    current_result = list(map(lambda x: x[key], mec_cascaded_eval_result_list))
    mec_cascaded_accuracy_map[key], mec_cascaded_kappa_map[key], mec_cascaded_accuracy_list[key], \
        mec_cascaded_kappa_list[key] = gather_final_metrics(current_result)

mec_cascaded_accuracy_map

{'auto-j': 83.09178743961351, 'judgelm': 54.10628019323672}

Decode information about the experiments

In [285]:
MEC_CASCADED_DESCRIPTION_MAP = {
    'auto-j': {
        'judge': 'Auto-J (CascadedEval with GPT-3.5) + MEC + BPC',
        'description': 'CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-3.5'
    },
    'judgelm': {
        'judge': 'JudgeLM (CascadedEval with GPT-3.5) + MEC + BPC',
        'description': 'CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-3.5'
    },
}

Gather everything in the final table (together with the previous steps)

In [286]:
mec_cascaded_final_table = pd.DataFrame({
    "judge": [v["judge"] for v in MEC_CASCADED_DESCRIPTION_MAP.values()],
    "description": [v["description"] for v in MEC_CASCADED_DESCRIPTION_MAP.values()],
    "accuracy": mec_cascaded_accuracy_map.values(),
    "kappa": mec_cascaded_kappa_map.values(),
}, index=mec_cascaded_accuracy_map.keys())

mec_cascaded_final_table = pd.concat([mec_cascaded_final_table, cascaded_final_table], ignore_index=True)
mec_cascaded_final_table

,judge,description,accuracy,kappa
0,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,83.091787,0.697283
1,JudgeLM (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,54.106280,0.227941
2,Auto-J (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,50.966184,0.211477
3,JudgeLM (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,56.763285,0.216264
4,GPT-3.5,EC (k = 1),44.166667,0.066273
5,GPT-4,EC (k = 1),88.333333,0.716802
6,GPT-3.5 + MEC + BPC,MEC (k = 3) + BPC (k = 3),79.444444,0.639853
7,GPT-4 + MEC + BPC,MEC (k = 3) + BPC (k = 3),46.944444,0.124227


# CascadeEval extraction with GPT-4 as a baseline
This chapter extracts evaluation data obtained using the **CascadedEval** approach, computes the control metrics (**accuracy** and **Cohen’s kappa**), and compiles a summary table for the **GPT-4 (baseline)** configuration.

Define folder of data type from merged sampling dataset

In [287]:
DATA_TYPE = 'vicuna-gpt4'

Obtain the aggregated sampled subset from the Vicuna dataset

In [288]:
vicuna_vanilla_questions_gpt4 = pd.read_json(
    f'../datasets/vicuna/sampled_data/cj_sampled_merged/vanilla-{DATA_TYPE}.json')
vicuna_vanilla_questions_gpt4['ground_truth'] = vicuna_vanilla_questions_gpt4['question_id'].apply(
    lambda x: text_lines[x - 1])
vicuna_vanilla_questions_gpt4 = vicuna_vanilla_questions_gpt4.set_index('question_id')

vicuna_vanilla_questions_gpt4

,text_vicuna,text_gpt,question_text,category,ground_truth
question_id,,,,,
7,Developing your critical thinking skills can h...,"Sure, here are a few practical tips to develop...",How can I develop my critical thinking skills?,generic,CHATGPT
17,"Vaccinations work by introducing a small, harm...",Vaccinations work by introducing a weakened or...,How do vaccinations work to protect individual...,knowledge,VICUNA13B
27,"Ladies and gentlemen, my name is [Chef's Name]...","Welcome, esteemed judges. I'm excited to prese...",Pretend to be a world-famous chef. How would y...,roleplay,CHATGPT
38,Observing the behavior of other people in a so...,Observing the behavior of other people in a so...,How can observing the behavior of other people...,common-sense,CHATGPT
45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,fermi,CHATGPT
57,The Suez Canal is a man-made waterway that con...,"If the Suez Canal had never been constructed, ...",What if the Suez Canal had never been construc...,counterfactual,VICUNA13B
66,Here is an implementation of a queue using two...,Here's an example implementation of a queue us...,Implement a queue data structure using two sta...,coding,CHATGPT
69,We can solve this equation using the following...,3x + 10 = 5(x - 2)\n\n3x + 10 = 5x - 10 (use ...,Solve for x in the equation 3x + 10 = 5(x - 2).,math,CHATGPT
76,Hello and welcome to our YouTube video on the ...,Title: The Evolution and Cultural Significance...,Write a script for a YouTube video exploring t...,writing,CHATGPT


Define the folder containing **CascadedEval** results for the **sampled portion** of the dataset for case when **GPT-4** is the baseline configuration.

In [289]:
cascaded_eval_result_list_gpt4 = load_cascaded_eval_results(
    cascad_eval_results_dir=CASCADED_EVAL_RESULTS,
    results_path=RESULTS_PATH,
    data_type=DATA_TYPE,
    vicuna_questions_template=vicuna_vanilla_questions_gpt4
)

cascaded_eval_result_list_gpt4[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,ground_truth,score
question_id,,,,,,
7,Developing your critical thinking skills can h...,"Sure, here are a few practical tips to develop...",How can I develop my critical thinking skills?,generic,CHATGPT,"[0, 1]"
17,"Vaccinations work by introducing a small, harm...",Vaccinations work by introducing a weakened or...,How do vaccinations work to protect individual...,knowledge,VICUNA13B,"[0, 1]"
27,"Ladies and gentlemen, my name is [Chef's Name]...","Welcome, esteemed judges. I'm excited to prese...",Pretend to be a world-famous chef. How would y...,roleplay,CHATGPT,"[9.0, 8.0]"
38,Observing the behavior of other people in a so...,Observing the behavior of other people in a so...,How can observing the behavior of other people...,common-sense,CHATGPT,"[10.0, 9.0]"
45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,fermi,CHATGPT,"[1, 0]"
57,The Suez Canal is a man-made waterway that con...,"If the Suez Canal had never been constructed, ...",What if the Suez Canal had never been construc...,counterfactual,VICUNA13B,"[0, 1]"
66,Here is an implementation of a queue using two...,Here's an example implementation of a queue us...,Implement a queue data structure using two sta...,coding,CHATGPT,"[10.0, 2.0]"
69,We can solve this equation using the following...,3x + 10 = 5(x - 2)\n\n3x + 10 = 5x - 10 (use ...,Solve for x in the equation 3x + 10 = 5(x - 2).,math,CHATGPT,"[10.0, 2.0]"
76,Hello and welcome to our YouTube video on the ...,Title: The Evolution and Cultural Significance...,Write a script for a YouTube video exploring t...,writing,CHATGPT,"[9.0, 8.5]"


Extract final judgement results and hit with the ground-truth


In [290]:
for key in cascaded_eval_result_list_gpt4[0].keys():
    for res in cascaded_eval_result_list_gpt4:
        res[key] = res[key].apply(lambda row: extract_result(row, text_lines), axis=1)

cascaded_eval_result_list_gpt4[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,ground_truth,score,result,result_hit
question_id,,,,,,,,
7,Developing your critical thinking skills can h...,"Sure, here are a few practical tips to develop...",How can I develop my critical thinking skills?,generic,CHATGPT,"[0, 1]",VICUNA13B,False
17,"Vaccinations work by introducing a small, harm...",Vaccinations work by introducing a weakened or...,How do vaccinations work to protect individual...,knowledge,VICUNA13B,"[0, 1]",VICUNA13B,True
27,"Ladies and gentlemen, my name is [Chef's Name]...","Welcome, esteemed judges. I'm excited to prese...",Pretend to be a world-famous chef. How would y...,roleplay,CHATGPT,"[9.0, 8.0]",CHATGPT,True
38,Observing the behavior of other people in a so...,Observing the behavior of other people in a so...,How can observing the behavior of other people...,common-sense,CHATGPT,"[10.0, 9.0]",CHATGPT,True
45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,fermi,CHATGPT,"[1, 0]",CHATGPT,True
57,The Suez Canal is a man-made waterway that con...,"If the Suez Canal had never been constructed, ...",What if the Suez Canal had never been construc...,counterfactual,VICUNA13B,"[0, 1]",VICUNA13B,True
66,Here is an implementation of a queue using two...,Here's an example implementation of a queue us...,Implement a queue data structure using two sta...,coding,CHATGPT,"[10.0, 2.0]",CHATGPT,True
69,We can solve this equation using the following...,3x + 10 = 5(x - 2)\n\n3x + 10 = 5x - 10 (use ...,Solve for x in the equation 3x + 10 = 5(x - 2).,math,CHATGPT,"[10.0, 2.0]",CHATGPT,True
76,Hello and welcome to our YouTube video on the ...,Title: The Evolution and Cultural Significance...,Write a script for a YouTube video exploring t...,writing,CHATGPT,"[9.0, 8.5]",CHATGPT,True


Extract mode of the result in the experiment results

In [291]:
cascaded_eval_result_list_gpt4 = aggregate_eval_results(cascaded_eval_result_list_gpt4)
cascaded_eval_result_list_gpt4[0]['auto-j']

,result,ground_truth,result_hit
question_id,,,
7,VICUNA13B,CHATGPT,False
17,VICUNA13B,VICUNA13B,True
27,CHATGPT,CHATGPT,True
38,CHATGPT,CHATGPT,True
45,CHATGPT,CHATGPT,True
57,VICUNA13B,VICUNA13B,True
66,CHATGPT,CHATGPT,True
69,CHATGPT,CHATGPT,True
76,CHATGPT,CHATGPT,True


Get accuracy and kappa for each of **40** repeated experiments

In [292]:
cascaded_accuracy_map_gpt4 = {}
cascaded_kappa_map_gpt4 = {}

for key in cascaded_eval_result_list_gpt4[0].keys():
    current_result = list(map(lambda x: x[key], cascaded_eval_result_list_gpt4))
    cascaded_accuracy_map_gpt4[key], cascaded_kappa_map_gpt4[key], _, _ = gather_final_metrics(
        current_result)

cascaded_accuracy_map_gpt4

{'auto-j': 86.71497584541062, 'judgelm': 77.05314009661838}

Decode information about the experiments

In [293]:
CASCADED_DESCRIPTION_GPT4_MAP = {
    'auto-j': {
        'judge': 'Auto-J (CascadedEval with GPT-4)',
        'description': 'CascadedEval + EC (k = 1) + GPT-4'
    },
    'judgelm': {
        'judge': 'JudgeLM (CascadedEval with GPT-4)',
        'description': 'CascadedEval + EC (k = 1) + GPT-4'
    },
}

Gather everything in the final table (together with the previous steps)

In [294]:
cascaded_gpt4_table = pd.DataFrame({
    "judge": [v["judge"] for v in CASCADED_DESCRIPTION_GPT4_MAP.values()],
    "description": [v["description"] for v in CASCADED_DESCRIPTION_GPT4_MAP.values()],
    "accuracy": cascaded_accuracy_map_gpt4.values(),
    "kappa": cascaded_kappa_map_gpt4.values(),
}, index=cascaded_accuracy_map_gpt4.keys())

cascaded_gpt4_table = pd.concat([mec_cascaded_final_table, cascaded_gpt4_table], ignore_index=True)
cascaded_gpt4_table

,judge,description,accuracy,kappa
0,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,83.091787,0.697283
1,JudgeLM (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,54.106280,0.227941
2,Auto-J (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,50.966184,0.211477
3,JudgeLM (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,56.763285,0.216264
4,GPT-3.5,EC (k = 1),44.166667,0.066273
5,GPT-4,EC (k = 1),88.333333,0.716802
6,GPT-3.5 + MEC + BPC,MEC (k = 3) + BPC (k = 3),79.444444,0.639853
7,GPT-4 + MEC + BPC,MEC (k = 3) + BPC (k = 3),46.944444,0.124227
8,Auto-J (CascadedEval with GPT-4),CascadedEval + EC (k = 1) + GPT-4,86.714976,0.692074
9,JudgeLM (CascadedEval with GPT-4),CascadedEval + EC (k = 1) + GPT-4,77.053140,0.534627


# CascadedEval extraction with GPT-4 with MEC
This chapter extracts evaluation data obtained using the **CascadedEval** approach, computes the control metrics (**accuracy** and **Cohen’s kappa**), and compiles a summary table for the **GPT-4 (MEC + BPC)** configuration.

Define folder of data type from merged sampling dataset

In [295]:
DATA_TYPE = 'vicuna-mec-gpt4'

Obtain the aggregated sampled subset from the Vicuna dataset

In [296]:
vicuna_mec_questions_gpt4 = pd.read_json(f'../datasets/vicuna/sampled_data/cj_sampled_merged/mec-bpc-vicuna-gpt4.json')
vicuna_mec_questions_gpt4['ground_truth'] = vicuna_mec_questions_gpt4['question_id'].apply(lambda x: text_lines[x - 1])
vicuna_mec_questions_gpt4 = vicuna_mec_questions_gpt4.set_index('question_id')

vicuna_mec_questions_gpt4

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth
question_id,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE


Extract final judgement results (either Auto-J / JudgeLM or MEC+BPC with GPT-4) for each question to calculate **accuracy** and **kappa**

In [297]:
mec_cascaded_eval_result_gpt4_list = load_cascaded_eval_results(
    cascad_eval_results_dir=CASCADED_EVAL_RESULTS,
    results_path=RESULTS_PATH,
    data_type=DATA_TYPE,
    vicuna_questions_template=vicuna_mec_questions_gpt4
)

mec_cascaded_eval_result_gpt4_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth,score
question_id,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[1, 0]"
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[1, 0]"
5,Certainly! Quantum computing is a type of comp...,Quantum computing is a type of computing that ...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[1, 0]"
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]"
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]"
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]"


Reverse rows that are under BPC

In [298]:
for key in mec_cascaded_eval_result_gpt4_list[0].keys():
    for res in mec_cascaded_eval_result_gpt4_list:
        res[key] = res[key].apply(reverse_row, axis=1)

mec_cascaded_eval_result_gpt4_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth,score
question_id,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]"
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]"
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]"
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]"
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]"


Extract final judgement results and hit with the ground-truth

In [299]:
for key in mec_cascaded_eval_result_gpt4_list[0].keys():
    for res in mec_cascaded_eval_result_gpt4_list:
        res[key] = res[key].apply(lambda row: extract_result(row, text_lines), axis=1)

mec_cascaded_eval_result_gpt4_list[0]['auto-j']

,text_vicuna,text_gpt,question_text,category,is_reversed,ground_truth,score,result,result_hit
question_id,,,,,,,,,
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,False,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]",VICUNA13B,True
5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,generic,True,VICUNA13B,"[0, 1]",VICUNA13B,True
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]",VICUNA13B,False
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]",VICUNA13B,False
14,Language and cultural barriers can have a sign...,Language and cultural barriers can have a sign...,How do language and cultural barriers affect t...,knowledge,False,TIE,"[0, 1]",VICUNA13B,False


Extract mode of the result in the experiment results

In [300]:
mec_cascaded_eval_result_gpt4_list = aggregate_eval_results(mec_cascaded_eval_result_gpt4_list)
mec_cascaded_eval_result_gpt4_list[0]['auto-j']

,result,ground_truth,result_hit
question_id,,,
5,VICUNA13B,VICUNA13B,True
14,VICUNA13B,TIE,False
29,VICUNA13B,VICUNA13B,True
37,VICUNA13B,VICUNA13B,True
47,VICUNA13B,CHATGPT,False
56,CHATGPT,TIE,False
63,CHATGPT,VICUNA13B,False
69,CHATGPT,CHATGPT,True
77,TIE,CHATGPT,False


Get accuracy and kappa for each of **40** repeated experiments

In [301]:
mec_cascaded_accuracy_gpt4_map = {}
mec_cascaded_kappa_gpt4_map = {}

mec_cascaded_accuracy_gpt4_list = {}
mec_cascaded_kappa_gpt4_list = {}

for key in mec_cascaded_eval_result_gpt4_list[0].keys():
    current_result = list(map(lambda x: x[key], mec_cascaded_eval_result_gpt4_list))
    mec_cascaded_accuracy_gpt4_map[key], mec_cascaded_kappa_gpt4_map[key], mec_cascaded_accuracy_gpt4_list[key], \
    mec_cascaded_kappa_gpt4_list[key] = gather_final_metrics(
        current_result)

mec_cascaded_accuracy_gpt4_map

{'auto-j': 49.275362318840564, 'judgelm': 58.69565217391303}

Decode information about the experiments

In [302]:
MEC_CASCADED_DESCRIPTION_GPT4_MAP = {
    'auto-j': {
        'judge': 'Auto-J (CascadedEval with GPT-4) + MEC + BPC',
        'description': 'CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-4'
    },
    'judgelm': {
        'judge': 'JudgeLM (CascadedEval with GPT-4) + MEC + BPC',
        'description': 'CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-4'
    },
}

Gather everything in the final table (together with the previous steps)


In [303]:
mec_cascaded_gpt_4table = pd.DataFrame({
    "judge": [v["judge"] for v in MEC_CASCADED_DESCRIPTION_GPT4_MAP.values()],
    "description": [v["description"] for v in MEC_CASCADED_DESCRIPTION_GPT4_MAP.values()],
    "accuracy": mec_cascaded_accuracy_gpt4_map.values(),
    "kappa": mec_cascaded_kappa_gpt4_map.values(),
}, index=mec_cascaded_accuracy_gpt4_map.keys())

mec_cascaded_gpt_4table = pd.concat([mec_cascaded_gpt_4table, cascaded_gpt4_table], ignore_index=True)
mec_cascaded_gpt_4table

,judge,description,accuracy,kappa
0,Auto-J (CascadedEval with GPT-4) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-4,49.275362,0.154104
1,JudgeLM (CascadedEval with GPT-4) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-4,58.695652,0.334435
2,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,83.091787,0.697283
3,JudgeLM (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,54.106280,0.227941
4,Auto-J (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,50.966184,0.211477
5,JudgeLM (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,56.763285,0.216264
6,GPT-3.5,EC (k = 1),44.166667,0.066273
7,GPT-4,EC (k = 1),88.333333,0.716802
8,GPT-3.5 + MEC + BPC,MEC (k = 3) + BPC (k = 3),79.444444,0.639853
9,GPT-4 + MEC + BPC,MEC (k = 3) + BPC (k = 3),46.944444,0.124227


# CJ test final split
Split final results to GPT-3.5-turbo and GPT-4

## GPT-3.5-turbo
- The combination of **MEC + BPC (k = 3)** leads to:
  - Statistically significant improvement in **accuracy** (p < 0.05)
  - Statistically significant improvement in **kappa correlation** (p < 0.05)

- Integrating **Auto-J** into the Calibrated Judge further improves performance:
  - **Accuracy increases to 83.1%**
  - **Kappa rises to 0.70**
  - Both metrics show statistically significant gains over MEC + BPC

- The results validate that **Calibrated Judge** contributes meaningfully to improving evaluation quality with GPT-3.5-Turbo.

In [304]:
gpt35_final = mec_cascaded_gpt_4table[mec_cascaded_gpt_4table['judge'].str.contains('GPT-3.5', na=False)]
gpt35_final

,judge,description,accuracy,kappa
2,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,83.091787,0.697283
3,JudgeLM (CascadedEval with GPT-3.5) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT...,54.106280,0.227941
4,Auto-J (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,50.966184,0.211477
5,JudgeLM (CascadedEval with GPT-3.5),CascadedEval + EC (k = 1) + GPT-3.5,56.763285,0.216264
6,GPT-3.5,EC (k = 1),44.166667,0.066273
8,GPT-3.5 + MEC + BPC,MEC (k = 3) + BPC (k = 3),79.444444,0.639853


## GPT-4

- Not directly evaluated in the same configuration based on the provided excerpt.
- No statistically significant performance results or comparisons reported for GPT-4 in this context.

In [305]:
gpt4_final = mec_cascaded_gpt_4table[mec_cascaded_gpt_4table['judge'].str.contains('GPT-4', na=False)]
gpt4_final

,judge,description,accuracy,kappa
0,Auto-J (CascadedEval with GPT-4) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-4,49.275362,0.154104
1,JudgeLM (CascadedEval with GPT-4) + MEC + BPC,CascadedEval + MEC (k = 3) + BPC (k = 3) + GPT-4,58.695652,0.334435
7,GPT-4,EC (k = 1),88.333333,0.716802
9,GPT-4 + MEC + BPC,MEC (k = 3) + BPC (k = 3),46.944444,0.124227
10,Auto-J (CascadedEval with GPT-4),CascadedEval + EC (k = 1) + GPT-4,86.714976,0.692074
11,JudgeLM (CascadedEval with GPT-4),CascadedEval + EC (k = 1) + GPT-4,77.053140,0.534627


# CJ results statistical testing

## GPT-4 accuracy tests

In [307]:
gpt_4_acc = sampled_acc_list['gpt4_k_1_bpc_0_t_1']
gpt_4_mec_acc = sampled_acc_list['gpt4_k_3_bpc_1_t_1']
calib_judge_lm_gpt4_acc = mec_cascaded_accuracy_gpt4_list['judgelm']
calib_auto_j_gpt4_acc = mec_cascaded_accuracy_gpt4_list['auto-j']

print("MEC vs baseline\n")
mec_tester = TwoSampleStatisticalTests(gpt_4_mec_acc, gpt_4_acc)
mec_tester.test_two_numerical_samples()

print("\nJudgeLM vs MEC\n")
mec_tester = TwoSampleStatisticalTests(calib_judge_lm_gpt4_acc, gpt_4_mec_acc)
mec_tester.test_two_numerical_samples(is_one_tailed=True)

print("\nAuto-J vs MEC\n")
mec_tester = TwoSampleStatisticalTests(gpt_4_mec_acc, calib_auto_j_gpt4_acc)
mec_tester.test_two_numerical_samples()

MEC vs baseline

The result of the p-value when checking the normality for the first dataset: 8.268180079220535e-05
The result of the p-value when checking the normality for the second dataset: 3.2512410563000877e-13
The result of the p-value when checking the variance uniform: 1.0646977938864353e-05
Mann-Whitney was chosen
Final p-value: 1.7316802045099713e-16
H0 has been rejected, Ha has been accepted

JudgeLM vs MEC

The result of the p-value when checking the normality for the first dataset: 4.463494043486819e-05
The result of the p-value when checking the normality for the second dataset: 8.268180079220535e-05
The result of the p-value when checking the variance uniform: 0.6277575704033395
Mann-Whitney was chosen
Final p-value: 5.996836639812117e-08
H0 has been rejected, Ha has been accepted

Auto-J vs MEC

The result of the p-value when checking the normality for the first dataset: 8.268180079220535e-05
The result of the p-value when checking the normality for the second dataset:

## GPT-4 kappa tests

In [308]:
gpt_4_kappa = sampled_kappa_list['gpt4_k_1_bpc_0_t_1']
gpt_4_mec_kappa = sampled_kappa_list['gpt4_k_3_bpc_1_t_1']
calib_judge_lm_gpt4_kappa = mec_cascaded_kappa_gpt4_list['judgelm']
calib_auto_j_gpt4_kappa = mec_cascaded_kappa_gpt4_list['auto-j']

print("MEC vs baseline\n")
mec_tester = TwoSampleStatisticalTests(gpt_4_mec_kappa, gpt_4_kappa)
mec_tester.test_two_numerical_samples()

print("\nJudgeLM vs MEC\n")
mec_tester = TwoSampleStatisticalTests(calib_judge_lm_gpt4_kappa, gpt_4_mec_kappa)
mec_tester.test_two_numerical_samples(is_one_tailed=True)

print("\nAuto-J vs MEC\n")
mec_tester = TwoSampleStatisticalTests(gpt_4_mec_kappa, calib_auto_j_gpt4_kappa)
mec_tester.test_two_numerical_samples()

MEC vs baseline

The result of the p-value when checking the normality for the first dataset: 0.0011599938206142493
The result of the p-value when checking the normality for the second dataset: 3.569372791683109e-13
The result of the p-value when checking the variance uniform: 6.875942870679636e-07
Mann-Whitney was chosen
Final p-value: 3.0062725535089836e-16
H0 has been rejected, Ha has been accepted

JudgeLM vs MEC

The result of the p-value when checking the normality for the first dataset: 0.0008171712808802247
The result of the p-value when checking the normality for the second dataset: 0.0011599938206142493
The result of the p-value when checking the variance uniform: 0.765572900146692
Mann-Whitney was chosen
Final p-value: 1.7285341079224916e-09
H0 has been rejected, Ha has been accepted

Auto-J vs MEC

The result of the p-value when checking the normality for the first dataset: 0.0011599938206142493
The result of the p-value when checking the normality for the second dataset: 0

## GPT-3.5-turbo accuracy tests

In [309]:
gpt_35_acc = sampled_acc_list['gpt35_k_1_bpc_0_t_1']
gpt_35_mec_acc = sampled_acc_list['gpt35_k_3_bpc_1_t_1']
calib_judge_lm_gpt35_acc = mec_cascaded_accuracy_list['judgelm']
calib_auto_j_gpt35_acc = mec_cascaded_accuracy_list['auto-j']

print("MEC vs baseline\n")
mec_tester = TwoSampleStatisticalTests(gpt_35_mec_acc, gpt_35_acc)
mec_tester.test_two_numerical_samples(is_one_tailed=True)

print("\nJudgeLM vs MEC\n")
mec_tester = TwoSampleStatisticalTests(calib_judge_lm_gpt35_acc, gpt_35_mec_acc)
mec_tester.test_two_numerical_samples()
print(calib_judge_lm_gpt35_acc.mean())
print(gpt_35_mec_acc.mean())

print("\nAuto-J vs MEC\n")
mec_tester = TwoSampleStatisticalTests(calib_auto_j_gpt35_acc, gpt_35_mec_acc)
mec_tester.test_two_numerical_samples()

MEC vs baseline

The result of the p-value when checking the normality for the first dataset: 5.977871270659952e-07
The result of the p-value when checking the normality for the second dataset: 1.1298991811008934e-05
The result of the p-value when checking the variance uniform: 0.11756183162228954
Mann-Whitney was chosen
Final p-value: 1.28089233092466e-15
H0 has been rejected, Ha has been accepted

JudgeLM vs MEC

The result of the p-value when checking the normality for the first dataset: 1.7372378393196735e-12
The result of the p-value when checking the normality for the second dataset: 5.977871270659952e-07
The result of the p-value when checking the variance uniform: 0.01594470270064864
Mann-Whitney was chosen
Final p-value: 1.3902772515143342e-17
H0 has been rejected, Ha has been accepted
54.10628019323672
79.44444444444444

Auto-J vs MEC

The result of the p-value when checking the normality for the first dataset: 4.618079521115577e-08
The result of the p-value when checking the

## GPT-3.5-turbo kappa tests

In [310]:
gpt_35_kappa = sampled_kappa_list['gpt35_k_1_bpc_0_t_1']
gpt_35_mec_kappa = sampled_kappa_list['gpt35_k_3_bpc_1_t_1']
calib_judge_lm_gpt35_kappa = mec_cascaded_kappa_list['judgelm']
calib_auto_j_gpt35_kappa = mec_cascaded_kappa_list['auto-j']

print("MEC vs baseline\n")
mec_tester = TwoSampleStatisticalTests(gpt_35_mec_kappa, gpt_35_kappa)
mec_tester.test_two_numerical_samples(is_one_tailed=True)

print("\nJudgeLM vs MEC\n")
mec_tester = TwoSampleStatisticalTests(calib_judge_lm_gpt35_kappa, gpt_35_mec_kappa)
mec_tester.test_two_numerical_samples()

print("\nAuto-J vs MEC\n")
mec_tester = TwoSampleStatisticalTests(calib_auto_j_gpt35_kappa, gpt_35_mec_kappa)
mec_tester.test_two_numerical_samples()

MEC vs baseline

The result of the p-value when checking the normality for the first dataset: 1.445952369266414e-05
The result of the p-value when checking the normality for the second dataset: 0.07888856378526432
The result of the p-value when checking the variance uniform: 0.008771681577263369
Mann-Whitney was chosen
Final p-value: 4.867435269562136e-15
H0 has been rejected, Ha has been accepted

JudgeLM vs MEC

The result of the p-value when checking the normality for the first dataset: 2.250507700088557e-12
The result of the p-value when checking the normality for the second dataset: 1.445952369266414e-05
The result of the p-value when checking the variance uniform: 0.001256070071085467
Mann-Whitney was chosen
Final p-value: 4.4077373778568324e-17
H0 has been rejected, Ha has been accepted

Auto-J vs MEC

The result of the p-value when checking the normality for the first dataset: 7.533003831714852e-08
The result of the p-value when checking the normality for the second dataset: 1.

# CS integration experiment result tests

This section documents the evaluation of the **Calibrated Scorer (CS)** integration test using **LC AlpacaEval** against the **pure CascadedEval** baseline.  
The experiments compare **accuracy** and **Cohen’s kappa correlation** between predicted results and ground truth across multiple judge configurations.  
Both **two-tailed** and **one-tailed** statistical tests are applied to determine significance.


Evaluation outputs were loaded from JSON files for the following judge configurations:

- **Auto-J (CascadedEval)** using GPT-3.5
- **Auto-J (CascadedEval)** using GPT-4
- **JudgeLM (CascadedEval)** using GPT-3.5
- **JudgeLM (CascadedEval)** using GPT-4

In [313]:
BASE_PATH = "../gathered_data/cj_cs_test_result"

In [314]:
alpaca_results = {
    'gpt-3-5': pd.read_json(f"{BASE_PATH}/alpaca-result-gpt-3.5.json"),
    'gpt-4': pd.read_json(f"{BASE_PATH}/alpaca-result-gpt-4.json"),
    'gpt-3-5-judgelm': pd.read_json(f"{BASE_PATH}/alpaca-result-gpt-3-5-judgelm.json"),
    'gpt-4-judgelm': pd.read_json(f"{BASE_PATH}/alpaca-result-gpt-4-judgelm.json"),
}

alpaca_results['gpt-3-5']

,question_id,model_response,baseline_response,instruction,ground_truth,result,preference,baseline_name,model_name,length_feature,predicted_result
0,5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986
1,18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986
2,26,"In the final seconds of the championship game,...",It's the final moments of the championship gam...,"As a sports commentator, describe the winning ...",CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.404011
3,40,In a world where automation is becoming increa...,It's important to strike a balance between job...,In a world where automation is becoming increa...,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986
4,45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.000899
...,...,...,...,...,...,...,...,...,...,...,...
355,45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.000899
356,53,"The Black Death, also known as the Plague, was...",If the Black Death had not occurred in the 14t...,What if the Black Death had not occurred in th...,CHATGPT,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986
357,63,Here is a regular expression that can be used ...,Here's a regular expression pattern in Python ...,Implement a regular expression in Python to va...,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.979749
358,70,The length of the line segment is the distance...,To find the length of a line segment with two ...,"If the endpoints of a line segment are (2, -2)...",CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.000899


Data was split into batches of size 9 and aggregated by **`question_id`** to compute majority-vote results

In [315]:
def result_extraction(predicted_result):
    if predicted_result >= 0.5:
        return 'VICUNA13B'

    return 'CHATGPT'


for name, value in alpaca_results.items():
    alpaca_results[name]['final_result'] = value['predicted_result'].apply(result_extraction)

alpaca_results['gpt-3-5']

,question_id,model_response,baseline_response,instruction,ground_truth,result,preference,baseline_name,model_name,length_feature,predicted_result,final_result
0,5,Quantum computing is a type of computing that ...,Certainly! Quantum computing is a type of comp...,Can you explain the basics of quantum computing?,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986,VICUNA13B
1,18,Social media platforms have revolutionized the...,Social media platforms play a significant role...,How do social media platforms influence the wa...,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986,VICUNA13B
2,26,"In the final seconds of the championship game,...",It's the final moments of the championship gam...,"As a sports commentator, describe the winning ...",CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.404011,CHATGPT
3,40,In a world where automation is becoming increa...,It's important to strike a balance between job...,In a world where automation is becoming increa...,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986,VICUNA13B
4,45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.000899,CHATGPT
...,...,...,...,...,...,...,...,...,...,...,...,...
355,45,"I'm sorry, but I cannot provide a specific ans...","According to the latest statistics, more than ...",How many text messages are sent globally in a ...,CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.000899,CHATGPT
356,53,"The Black Death, also known as the Plague, was...",If the Black Death had not occurred in the 14t...,What if the Black Death had not occurred in th...,CHATGPT,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.998986,VICUNA13B
357,63,Here is a regular expression that can be used ...,Here's a regular expression pattern in Python ...,Implement a regular expression in Python to va...,VICUNA13B,VICUNA13B,0,vicuna13b,gpt-3.5-turbo,0,0.979749,VICUNA13B
358,70,The length of the line segment is the distance...,To find the length of a line segment with two ...,"If the endpoints of a line segment are (2, -2)...",CHATGPT,CHATGPT,1,vicuna13b,gpt-3.5-turbo,0,0.000899,CHATGPT


In [316]:
for name, value in alpaca_results.items():
    answer_list = [value[i:i + 9] for i in range(0, len(value), 9)]

    alpaca_results[name] = answer_list

len(alpaca_results['gpt-3-5'])

40

Extract final result hits for both cases

In [317]:
for key in alpaca_results.keys():
    for idx, df in enumerate(alpaca_results[key]):
        df = df.groupby('question_id').agg({
            'final_result': lambda x: x.value_counts().idxmax(),
            'result': lambda x: x.value_counts().idxmax(),
            'ground_truth': 'first',
        })

        df['result_hit'] = df['result'] == df['ground_truth']
        df['final_result_hit'] = df['final_result'] == df['ground_truth']

        alpaca_results[key][idx] = df

alpaca_results['gpt-3-5'][0]

,final_result,result,ground_truth,result_hit,final_result_hit
question_id,,,,,
5,VICUNA13B,VICUNA13B,VICUNA13B,True,True
18,VICUNA13B,VICUNA13B,VICUNA13B,True,True
26,CHATGPT,CHATGPT,CHATGPT,True,True
40,VICUNA13B,VICUNA13B,VICUNA13B,True,True
45,CHATGPT,CHATGPT,CHATGPT,True,True
53,VICUNA13B,VICUNA13B,CHATGPT,False,False
63,VICUNA13B,VICUNA13B,VICUNA13B,True,True
70,CHATGPT,CHATGPT,CHATGPT,True,True
76,CHATGPT,CHATGPT,CHATGPT,True,True


Get accuracy and kappa for each of **40** repeated experiments for CJ only

In [318]:
cascaded_acc = {}
cascaded_kappa = {}

cascaded_acc_lists = {}
cascaded_kappa_lists = {}

for key, current_list in alpaca_results.items():
    cascaded_acc[key], cascaded_kappa[key], cascaded_acc_lists[key], cascaded_kappa_lists[key] = gather_final_metrics(current_list)

cascaded_acc

{'gpt-3-5': 83.05555555555556,
 'gpt-4': 64.72222222222221,
 'gpt-3-5-judgelm': 76.11111111111111,
 'gpt-4-judgelm': 60.27777777777777}

Get accuracy and kappa for each of **40** repeated experiments for CJ + CS

In [319]:
lc_acc = {}
lc_kappa = {}

lc_acc_lists = {}
lc_kappa_lists = {}

for key, current_list in alpaca_results.items():
    lc_acc[key], lc_kappa[key], lc_acc_lists[key], lc_kappa_lists[key] = gather_final_metrics(
        current_list, result_feature="final_result",
        hit_feature="final_result_hit")

lc_acc

{'gpt-3-5': 88.88888888888889,
 'gpt-4': 66.66666666666666,
 'gpt-3-5-judgelm': 77.77777777777779,
 'gpt-4-judgelm': 55.55555555555556}

Only GPT-4 results are interesting and H_0 was not rejected => it is impossible to state that there is a difference with and without CS 

In [327]:
print("TWO-TAILED\n")

for key, current_list in alpaca_results.items():
    print("\n" + key)

    print("ACC\n")
    tester = TwoSampleStatisticalTests(lc_acc_lists[key], cascaded_acc_lists[key])
    tester.test_two_numerical_samples()

    print("\nKAPPA\n")
    kappa_tester = TwoSampleStatisticalTests(lc_kappa_lists[key], cascaded_kappa_lists[key])
    kappa_tester.test_two_numerical_samples()

TWO-TAILED


gpt-3-5
ACC

The result of the p-value when checking the normality for the first dataset: 1.0
The result of the p-value when checking the normality for the second dataset: 1.1222766811463214e-07
The result of the p-value when checking the variance uniform: 7.745914273667814e-34
Mann-Whitney was chosen
Final p-value: 3.136190554190501e-07
H0 has been rejected, Ha has been accepted

KAPPA

The result of the p-value when checking the normality for the first dataset: 1.0
The result of the p-value when checking the normality for the second dataset: 1.4727221859920516e-07
The result of the p-value when checking the variance uniform: 3.280117188399829e-35
Mann-Whitney was chosen
Final p-value: 3.2901117077849037e-07
H0 has been rejected, Ha has been accepted

gpt-4
ACC

The result of the p-value when checking the normality for the first dataset: 1.0
The result of the p-value when checking the normality for the second dataset: 3.3940281146300214e-05
The result of the p-value when 

/opt/homebrew/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


Only GPT-3.5-turbo results are interesting and H_0 was not rejected => it is possible to state that CS component adjust the results

In [322]:
print("ONE-TAILED\n")

for key, current_list in alpaca_results.items():
    print(key)

    print("\nACC\n")
    tester = TwoSampleStatisticalTests(lc_acc_lists[key], cascaded_acc_lists[key])
    tester.test_two_numerical_samples(is_one_tailed=True)

    print("\nKAPPA\n")
    kappa_tester = TwoSampleStatisticalTests(lc_kappa_lists[key], cascaded_kappa_lists[key])
    kappa_tester.test_two_numerical_samples(is_one_tailed=True)

ONE-TAILED

gpt-3-5

ACC

The result of the p-value when checking the normality for the first dataset: 1.0
The result of the p-value when checking the normality for the second dataset: 1.1222766811463214e-07
The result of the p-value when checking the variance uniform: 7.745914273667814e-34
Mann-Whitney was chosen
Final p-value: 1.5680952770952505e-07
H0 has been rejected, Ha has been accepted

KAPPA

The result of the p-value when checking the normality for the first dataset: 1.0
The result of the p-value when checking the normality for the second dataset: 1.4727221859920516e-07
The result of the p-value when checking the variance uniform: 3.280117188399829e-35
Mann-Whitney was chosen
Final p-value: 1.6450558538924518e-07
H0 has been rejected, Ha has been accepted
gpt-4

ACC

The result of the p-value when checking the normality for the first dataset: 1.0
The result of the p-value when checking the normality for the second dataset: 3.3940281146300214e-05
The result of the p-value when

/opt/homebrew/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


Decode information about the experiments

In [323]:
LC_MAP = {
    'gpt-3-5': {
        'judge': 'Auto-J (CascadedEval with GPT-3.5) + MEC + BPC',
        'description': 'LC AlpacaEval test'
    },
    'gpt-4': {
        'judge': 'Auto-J (CascadedEval with GPT-4) + MEC + BPC',
        'description': 'LC AlpacaEval test'
    },
    'gpt-3-5-judgelm': {
        'judge': 'JudgeLm (CascadedEval with GPT-3.5) + MEC + BPC',
        'description': 'LC AlpacaEval test'
    },
    'gpt-4-judgelm': {
        'judge': 'JudgeLm (CascadedEval with GPT-4) + MEC + BPC',
        'description': 'LC AlpacaEval test'
    },
}

CASCADED_MAP = {
    'gpt-3-5': {
        'judge': 'Auto-J (CascadedEval with GPT-3.5) + MEC + BPC',
        'description': 'Pure result'
    },
    'gpt-4': {
        'judge': 'Auto-J (CascadedEval with GPT-4) + MEC + BPC',
        'description': 'Pure result'
    },
    'gpt-3-5-judgelm': {
        'judge': 'JudgeLm (CascadedEval with GPT-3.5) + MEC + BPC',
        'description': 'Pure result'
    },
    'gpt-4-judgelm': {
        'judge': 'JudgeLm (CascadedEval with GPT-4) + MEC + BPC',
        'description': 'Pure result'
    },
}

Gather everything in the final table

In [324]:
lc_table = pd.DataFrame({
    "judge": [v["judge"] for v in LC_MAP.values()],
    "description": [v["description"] for v in LC_MAP.values()],
    "accuracy": lc_acc.values(),
    "kappa": lc_kappa.values(),
}, index=lc_acc.keys())

cascaded_table = pd.DataFrame({
    "judge": [v["judge"] for v in CASCADED_MAP.values()],
    "description": [v["description"] for v in CASCADED_MAP.values()],
    "accuracy": cascaded_acc.values(),
    "kappa": cascaded_kappa.values(),
}, index=cascaded_acc.keys())

lc_table = pd.concat([lc_table, cascaded_table])
lc_table

,judge,description,accuracy,kappa
gpt-3-5,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,LC AlpacaEval test,88.888889,0.780488
gpt-4,Auto-J (CascadedEval with GPT-4) + MEC + BPC,LC AlpacaEval test,66.666667,0.181818
gpt-3-5-judgelm,JudgeLm (CascadedEval with GPT-3.5) + MEC + BPC,LC AlpacaEval test,77.777778,0.571429
gpt-4-judgelm,JudgeLm (CascadedEval with GPT-4) + MEC + BPC,LC AlpacaEval test,55.555556,0.000000
gpt-3-5,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,Pure result,83.055556,0.670439
gpt-4,Auto-J (CascadedEval with GPT-4) + MEC + BPC,Pure result,64.722222,0.174301
gpt-3-5-judgelm,JudgeLm (CascadedEval with GPT-3.5) + MEC + BPC,Pure result,76.111111,0.539231
gpt-4-judgelm,JudgeLm (CascadedEval with GPT-4) + MEC + BPC,Pure result,60.277778,0.132330


# CS integration experiment results splitting
Split results on GPT-3.5 and GPT-4

## GPT-3.5-turbo

In [328]:
gpt35_final = lc_table[lc_table['judge'].str.contains('GPT-3.5', na=False)]
gpt35_final

,judge,description,accuracy,kappa
gpt-3-5,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,LC AlpacaEval test,88.888889,0.780488
gpt-3-5-judgelm,JudgeLm (CascadedEval with GPT-3.5) + MEC + BPC,LC AlpacaEval test,77.777778,0.571429
gpt-3-5,Auto-J (CascadedEval with GPT-3.5) + MEC + BPC,Pure result,83.055556,0.670439
gpt-3-5-judgelm,JudgeLm (CascadedEval with GPT-3.5) + MEC + BPC,Pure result,76.111111,0.539231


## GPT-4

In [329]:
gpt4_final = lc_table[lc_table['judge'].str.contains('GPT-4', na=False)]
gpt4_final

,judge,description,accuracy,kappa
gpt-4,Auto-J (CascadedEval with GPT-4) + MEC + BPC,LC AlpacaEval test,66.666667,0.181818
gpt-4-judgelm,JudgeLm (CascadedEval with GPT-4) + MEC + BPC,LC AlpacaEval test,55.555556,0.000000
gpt-4,Auto-J (CascadedEval with GPT-4) + MEC + BPC,Pure result,64.722222,0.174301
gpt-4-judgelm,JudgeLm (CascadedEval with GPT-4) + MEC + BPC,Pure result,60.277778,0.132330


# LC verbosity dataset preparation

In [ ]:
dirs = [name for name in os.listdir("./verbosity_datasets") if
        os.path.isdir(os.path.join("./verbosity_datasets", name))]
dirs.sort()
dirs

In [ ]:
BASELINE_NAME = "gpt4_1106_preview"
baseline_outputs = pd.read_json(f"./verbosity_datasets/{BASELINE_NAME}/model_outputs.json")
baseline_outputs

Define the random 10 questions per dataset

In [ ]:
model_outputs = pd.read_json(f"./verbosity_datasets/{dirs[0]}/model_outputs.json")
model_outputs

In [ ]:
instructions = model_outputs.groupby('dataset', group_keys=False).sample(n=10, random_state=42)['instruction'].unique()
len(instructions)

In [ ]:
judgement_objects = []

for name in dirs:
    current_outputs = pd.read_json(f"./verbosity_datasets/{name}/model_outputs.json")

    for instruction in instructions:
        judgement_objects.append({
            'instruction': instruction,
            'answer_1': current_outputs[current_outputs['instruction'] == instruction]['output'].iloc[0],
            'answer_2': baseline_outputs[baseline_outputs['instruction'] == instruction]['output'].iloc[0],
            'model_1': name,
            'model_2': BASELINE_NAME
        })

len(judgement_objects)

In [ ]:
judgement_df = pd.DataFrame(judgement_objects)
judgement_df

In [ ]:
rows = []

for _, row in judgement_df.iterrows():
    for _ in range(3):
        rows.append(row.copy())

    swapped_row = row.copy()
    swapped_row['answer_1'], swapped_row['answer_2'] = row['answer_2'], row['answer_1']
    swapped_row['model_1'], swapped_row['model_2'] = row['model_2'], row['model_1']
    for _ in range(3):
        rows.append(swapped_row.copy())

df_judgement_expanded = pd.DataFrame(rows).reset_index(drop=True)
df_judgement_expanded.to_json("./verbosity_dataset/judgement_expanded.json", orient='records')
df_judgement_expanded

# LC verbosity state-of-the-art separation

In [ ]:
with open("./relia_scores/auto-j/verbosity-relia.json", 'r') as f:
    verbosity_relia_scores = json.load(f)

verbosity_relia_scores = verbosity_relia_scores['Entropy']
len(verbosity_relia_scores)

In [ ]:
verbosity_fine_tune_evaluation = []
with open("./relia_scores/auto-j/verbosity-logit.jsonl", 'r') as f:
    for line in f:
        verbosity_fine_tune_evaluation.append(json.loads(line))

len(verbosity_fine_tune_evaluation)

In [ ]:
df_judgement_expanded['score'] = verbosity_fine_tune_evaluation
df_judgement_expanded['entropy'] = verbosity_relia_scores

df_judgement_expanded

In [ ]:
non_confident_indices = extract_non_confident_indices(df_judgement_expanded)
len(non_confident_indices)

In [ ]:
state_judgement_df = df_judgement_expanded.loc[list(non_confident_indices)]
state_judgement_df

In [ ]:
gpt_instructions = pd.DataFrame({})

gpt_instructions['question_id'] = gpt_instructions['text'] = state_judgement_df['instruction']
gpt_instructions = gpt_instructions.reset_index()
gpt_instructions.to_json("./verbosity_datasets/questions.jsonl", lines=True, orient='records')
gpt_instructions

In [ ]:
gpt_first = pd.DataFrame({})

gpt_first['question_id'] = state_judgement_df['instruction']
gpt_first['text'] = state_judgement_df['answer_1']

gpt_first = gpt_first.reset_index()

gpt_first.to_json("./verbosity_datasets/first_answer.jsonl", lines=True, orient='records')
gpt_first

In [ ]:
gpt_second = pd.DataFrame({})

gpt_second['question_id'] = state_judgement_df['instruction']
gpt_second['text'] = state_judgement_df['answer_2']

gpt_second = gpt_second.reset_index()

gpt_second.to_json("./verbosity_datasets/second_answer.jsonl", lines=True, orient='records')
gpt_second

# Analysis application

In [ ]:
final_judgement = pd.read_json("./verbosity_datasets/final_judgement.json").set_index('index')
final_judgement

In [ ]:
final_combination_df = pd.read_json("./verbosity_datasets/final_combination.json")

final_combination_df

In [ ]:
swapped_df = final_combination_df.copy()

for i in range(0, len(swapped_df), 6):
    # Select the next 3 rows (i+3 to i+6, exclusive)
    swap_slice = swapped_df.iloc[i + 3:i + 6].copy()

    # Swap model_1 and model_2
    swapped_df.iloc[i + 3:i + 6, swapped_df.columns.get_loc('model_1')] = swap_slice['model_2'].values
    swapped_df.iloc[i + 3:i + 6, swapped_df.columns.get_loc('model_2')] = swap_slice['model_1'].values

    swapped_scores = swap_slice['score'].apply(lambda x: [x[1], x[0]]).values
    swapped_df.iloc[i + 3:i + 6, swapped_df.columns.get_loc('score')] = swapped_scores

swapped_df

In [ ]:
swapped_df['result'] = swapped_df.apply(
    lambda row: row['model_1'] if row['score'][0] >= row['score'][1] else row['model_2'],
    axis=1
)

swapped_df

In [ ]:
agg_dict = {col: (lambda x: x.value_counts().idxmax()) if col == 'result' else 'first'
            for col in swapped_df.columns}

grouped_final_combination = swapped_df.groupby(['instruction', 'model_1', 'model_2']).agg(agg_dict).drop(
    ['instruction', 'score', 'entropy', 'model_1', 'model_2'], axis=1)

grouped_final_combination

In [ ]:
df = grouped_final_combination.copy()

all_models = pd.unique(df.index.get_level_values('model_1'))

# Count how many times each model appears as the result (wins)
wins = df['result'].value_counts().drop('gpt4_1106_preview').reindex(all_models, fill_value=0)

# Compute win rate
win_rate = (wins / df.index.get_level_values('instruction').nunique() * 100).sort_values(ascending=False)

# Optional: convert to DataFrame for inspection
win_rate_df = win_rate.reset_index()
win_rate_df.columns = ['model', 'win_rate']
win_rate_df

# LC dataset preparation for verbosity

In [ ]:
DROP_LIST = ['alpaca-7b_verbose', 'alpaca-7b', 'alpaca-7b_concise', 'gpt4_0613_verbose', 'gpt4_0613_concise',
             'gpt4_0613']

In [ ]:
dropped_df = grouped_final_combination[
    ~grouped_final_combination.index.get_level_values('model_1').isin(DROP_LIST) &
    ~grouped_final_combination.index.get_level_values('model_2').isin(DROP_LIST)
    ]

dropped_df

In [ ]:
renamed_df = dropped_df.reset_index()
renamed_df = renamed_df.rename(columns={
    'model_1': 'model_name',
    'model_2': 'baseline_name',
    'answer_1': 'model_response',
    'answer_2': 'baseline_response'
})
renamed_df

In [ ]:
def alternating_zeros_ones():
    while True:
        yield 0
        yield 1


gen = alternating_zeros_ones()


def extract_preferences(row):
    if row['model_name'] != BASELINE_NAME:
        return 1 if row['result'] != BASELINE_NAME else 0
    else:
        next_gen = next(gen)
        return next_gen


final_verb_df = renamed_df.copy()

final_verb_df['preference'] = final_verb_df.apply(lambda row: extract_preferences(row), axis=1)
final_verb_df

In [ ]:
final_verb_df.to_json("./verbosity_datasets/final_verbs.json", orient='records')

In [ ]:
balanced_rows = []

for (baseline, model), group in final_verb_df.groupby(['baseline_name', 'model_name']):
    group_0 = group[group['preference'] == 0]
    group_1 = group[group['preference'] == 1]

    more_less = len(group_0) > len(group_1)
    n = min(len(group_0), len(group_1))

    if n == 0:
        continue

    sampled_0 = group_0.sample(n=n if not more_less else min(int(4 * n), len(group_0)), random_state=42)
    sampled_1 = group_1.sample(n=n if more_less else min(int(4 * n), len(group_1)), random_state=42)

    balanced = pd.concat([sampled_0, sampled_1])
    balanced_rows.append(balanced)

balanced_df = pd.concat(balanced_rows).reset_index(drop=True)
balanced_df.to_json('./verbosity_datasets/balanced.json', orient='records')
balanced_df

# Verbosity LC check

In [ ]:
verbosity_lc_results = pd.read_json("./verbosity_datasets/alpaca-result-verbosity.json")
verbosity_lc_results

In [ ]:
verbosity_lc_results['final_result'] = verbosity_lc_results.apply(
    lambda row: row['baseline_name']
    if row['predicted_result'] >= 0.5 else row['model_name']
    if row['model_name'] != BASELINE_NAME
    else row['model_name'] + 'MODEL',
    axis=1
)
verbosity_lc_results

In [ ]:
agg_dict = {col: (lambda x: x.value_counts().idxmax()) if col == 'final_result' else 'first'
            for col in verbosity_lc_results.columns}

lc_final_combination = verbosity_lc_results.groupby(['instruction', 'model_name', 'baseline_name']).agg(agg_dict)

lc_final_combination

In [ ]:
lc_final_combination.loc[
    lc_final_combination['final_result'] == 'gpt4_1106_previewMODEL',
    'model_name'
] = 'gpt4_1106_previewMODEL'

lc_final_combination

In [ ]:
lc_df = lc_final_combination.copy()

lc_all_models = pd.unique(lc_df['model_name'].unique())

# Count how many times each model appears as the result (wins)
lc_wins = lc_df['final_result'].value_counts().drop(BASELINE_NAME)

# Compute win rate
lc_win_rate = (lc_wins / lc_df.index.get_level_values('instruction').nunique() * 100).sort_values(ascending=False)

# Optional: convert to DataFrame for inspection
lc_win_rate_df = lc_win_rate.reset_index()
lc_win_rate_df.columns = ['model', 'win_rate']
lc_win_rate_df